# Crop Recommendation Model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load and Explore Data

In [ ]:
df = pd.read_csv("../data/Crop_recommendation.csv")
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
df.describe()

In [ ]:
print("Crop Distribution:")
print(df['label'].value_counts())

plt.figure(figsize=(14, 6))
df['label'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Distribution of Crop Types', fontsize=16, fontweight='bold')
plt.xlabel('Crop Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
df_enhanced = df.copy()

# NPK ratio features
df_enhanced['NPK_sum'] = df_enhanced['N'] + df_enhanced['P'] + df_enhanced['K']
df_enhanced['NP_ratio'] = df_enhanced['N'] / (df_enhanced['P'] + 1)
df_enhanced['NK_ratio'] = df_enhanced['N'] / (df_enhanced['K'] + 1)
df_enhanced['PK_ratio'] = df_enhanced['P'] / (df_enhanced['K'] + 1)

# Environmental interaction features
df_enhanced['temp_humidity_interaction'] = df_enhanced['temperature'] * df_enhanced['humidity']
df_enhanced['temp_rainfall_interaction'] = df_enhanced['temperature'] * df_enhanced['rainfall']
df_enhanced['humidity_rainfall_interaction'] = df_enhanced['humidity'] * df_enhanced['rainfall']

print(f"Enhanced features created! Total features: {df_enhanced.shape[1]}")
df_enhanced.head()

## Data Preprocessing

In [ ]:
X = df_enhanced.drop(['label'], axis=1)
y = df_enhanced['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preprocessing completed!")

## Model Training - XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train_scaled, y_train)

y_train_pred_xgb = xgb_model.predict(X_train_scaled)
y_test_pred_xgb = xgb_model.predict(X_test_scaled)

train_acc_xgb = accuracy_score(y_train, y_train_pred_xgb)
test_acc_xgb = accuracy_score(y_test, y_test_pred_xgb)
test_f1_xgb = f1_score(y_test, y_test_pred_xgb, average='weighted')

print(f"XGBoost Train Accuracy: {train_acc_xgb:.4f}")
print(f"XGBoost Test Accuracy: {test_acc_xgb:.4f}")
print(f"XGBoost Test F1-Score: {test_f1_xgb:.4f}")

cv_scores_xgb = cross_val_score(xgb_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"\nCross-Validation Scores: {cv_scores_xgb}")
print(f"Mean CV Accuracy: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")

## Model Training - LightGBM

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_model.fit(X_train_scaled, y_train)
y_test_pred_lgbm = lgbm_model.predict(X_test_scaled)

test_acc_lgbm = accuracy_score(y_test, y_test_pred_lgbm)
test_f1_lgbm = f1_score(y_test, y_test_pred_lgbm, average='weighted')

print(f"LightGBM Test Accuracy: {test_acc_lgbm:.4f}")
print(f"LightGBM Test F1-Score: {test_f1_lgbm:.4f}")

## Ensemble Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

ensemble_model = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('rf', rf_model)
    ],
    voting='soft',
    n_jobs=-1
)

ensemble_model.fit(X_train_scaled, y_train)
y_test_pred_ensemble = ensemble_model.predict(X_test_scaled)

test_acc_ensemble = accuracy_score(y_test, y_test_pred_ensemble)
test_f1_ensemble = f1_score(y_test, y_test_pred_ensemble, average='weighted')

print(f"Ensemble Test Accuracy: {test_acc_ensemble:.4f}")
print(f"Ensemble Test F1-Score: {test_f1_ensemble:.4f}")

## Model Comparison

In [ ]:
model_comparison = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Ensemble'],
    'Test Accuracy': [test_acc_xgb, test_acc_lgbm, test_acc_ensemble],
    'Test F1-Score': [test_f1_xgb, test_f1_lgbm, test_f1_ensemble]
})

print("\nModel Comparison:")
print(model_comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(model_comparison['Model'], model_comparison['Test Accuracy'], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_ylim([0.95, 1.0])
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(model_comparison['Model'], model_comparison['Test F1-Score'], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Model F1-Score Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('F1-Score', fontsize=12)
axes[1].set_ylim([0.95, 1.0])
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Detailed Evaluation

In [ ]:
best_model = xgb_model
y_test_pred = y_test_pred_xgb

print("Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=le.classes_, digits=4))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - XGBoost Model', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Feature Importance Analysis

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10))

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['Feature'].head(15), feature_importance['Importance'].head(15), color='steelblue')
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## SHAP Analysis

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, plot_type="bar", show=False)
plt.title('SHAP Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Save Models

In [ ]:
import joblib
import os

os.makedirs('../artifacts', exist_ok=True)

joblib.dump(best_model, '../artifacts/model.joblib')
joblib.dump(scaler, '../artifacts/scaler.joblib')
joblib.dump(le, '../artifacts/label_encoder.joblib')
joblib.dump(ensemble_model, '../artifacts/model_ensemble.joblib')
joblib.dump(X.columns.tolist(), '../artifacts/feature_names.joblib')

print("Models saved successfully!")

## Performance Summary

In [ ]:
print("="*60)
print("CROP RECOMMENDATION MODEL - PERFORMANCE SUMMARY")
print("="*60)
print(f"\nDataset Size: {len(df)} samples")
print(f"Number of Crops: {len(le.classes_)}")
print(f"Number of Features: {X.shape[1]}")
print(f"\nBest Model: XGBoost")
print(f"Test Accuracy: {test_acc_xgb:.4f} ({test_acc_xgb*100:.2f}%)")
print(f"Test F1-Score: {test_f1_xgb:.4f}")
print(f"Cross-Validation Accuracy: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")
print(f"\nEnsemble Model Performance:")
print(f"Test Accuracy: {test_acc_ensemble:.4f} ({test_acc_ensemble*100:.2f}%)")
print(f"Test F1-Score: {test_f1_ensemble:.4f}")
print("="*60)